BI-DIRECTIONAL LSTM CLASSIFIER MODEL USING OHLCV DATA

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils import class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# 1. Reproducibility
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(42)

# 2. Load & Advanced Feature Engineering
df = pd.read_csv('NIFTY50.csv').dropna()
# Convert to returns to make data stationary (crucial for LSTMs)
df['Open_Ret'] = df['Open'].pct_change()
df['High_Ret'] = df['High'].pct_change()
df['Low_Ret'] = df['Low'].pct_change()
df['Close_Ret'] = df['Close'].pct_change()
df['Vol_Ret'] = df['Volume'].replace(0, 1).pct_change()

# Binary Target: 1 if next day goes up, 0 if down
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.dropna().replace([np.inf, -np.inf], 0)

features = ['Open_Ret', 'High_Ret', 'Low_Ret', 'Close_Ret', 'Vol_Ret']
X_scaled = StandardScaler().fit_transform(df[features])

# 3. Create Sequences (10-day lookback)
def create_sequences(data, target, seq_length=100):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(target[i + seq_length])
    return np.array(X), np.array(y)

X_lstm, y_lstm = create_sequences(X_scaled, df['Target'].values)

# 4. Time-based Split
train_split = int(len(X_lstm) * 0.8)
val_split = int(len(X_lstm) * 0.9)

X_train, y_train = X_lstm[:train_split], y_lstm[:train_split]
X_val, y_val = X_lstm[train_split:val_split], y_lstm[train_split:val_split]
X_test, y_test = X_lstm[val_split:], y_lstm[val_split:]


weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(enumerate(weights))

# 5. Deep Bi-LSTM Model
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True, input_shape=(10, len(features)))),
    BatchNormalization(),
    Dropout(0.3),

    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),

    Bidirectional(LSTM(32)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# 6. Training with Early Stopping
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    class_weight=cw_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=1
)

# 7. Results
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['Down', 'Up']))

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


51/51 ━━━━━━━━━━━━━━━━━━━━ 51s 704ms/step - accuracy: 0.5099 - loss: 0.7466 - val_accuracy: 0.5199 - val_loss: 0.6932
Epoch 2/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 37s 730ms/step - accuracy: 0.5120 - loss: 0.7256 - val_accuracy: 0.4776 - val_loss: 0.6946
Epoch 3/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 36s 703ms/step - accuracy: 0.4928 - loss: 0.7231 - val_accuracy: 0.4453 - val_loss: 0.6974
Epoch 4/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 40s 682ms/step - accuracy: 0.4964 - loss: 0.7158 - val_accuracy: 0.4527 - val_loss: 0.6979
Epoch 5/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 41s 679ms/step - accuracy: 0.5112 - loss: 0.7055 - val_accuracy: 0.4627 - val_loss: 0.7045
Epoch 6/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 37s 717ms/step - accuracy: 0.5238 - loss: 0.7036 - val_accuracy: 0.4627 - val_loss: 0.7033
Epoch 7/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 41s 729ms/step - accuracy: 0.5382 - loss: 0.6986 - val_accuracy: 0.4552 - val_loss: 0.7098
Epoch 8/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 40s 714ms/step - accuracy: 0.5396 - loss: 0.7021 - val_accuracy